# Lab 1 — AI/ML map, problem framing và prediction contract

**Thời lượng:** 90 phút · **Case:** support ticket triage · **Nguyên tắc:** decision trước model.

## Mục tiêu và đầu ra

Bạn sẽ phân loại tám scenario, frame bốn bài toán, khóa Canvas/Contract, audit feature availability và lưu năm artifact vào `outputs/`. Chạy notebook từ đầu đến cuối; mọi assertion cuối phải pass.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
print(f'Project root: {ROOT}')

## 1. Bản đồ khái niệm

`AI ⊃ ML ⊃ Deep Learning`; TensorFlow/Keras là công cụ, không phải task family. Supervised cần nhãn; unsupervised không có target định trước; generative tạo nội dung mới. Classification tạo category, regression tạo số, clustering tạo nhóm, ranking tạo thứ tự.

In [ ]:
scenarios = pd.read_csv(DATA / 'scenario_cards.csv')
display(scenarios[['scenario_id', 'title', 'non_ml_goal', 'decision', 'target_kind', 'labels_available', 'actionable']])

### Dừng 10 phút

Trước khi chạy cell kế tiếp, tự ghi task family và lý do cho cả tám scenario. S05 có quy tắc thuế đầy đủ/ổn định nên dùng rule engine; S07 không có action nên chưa sẵn sàng.

In [ ]:
def task_family(row: pd.Series) -> str:
    if not bool(row['actionable']):
        return 'not_ready_no_action'
    if row['scenario_id'] == 'S05':
        return 'non_ml_rule_engine'
    mapping = {
        'binary': 'classification',
        'numeric': 'regression',
        'clusters': 'clustering',
        'generated_content': 'generation',
        'ordered_list': 'ranking',
    }
    return mapping[row['target_kind']]

framed = scenarios.copy()
framed['task_family'] = framed.apply(task_family, axis=1)
framed['ml_suitable'] = ~framed['task_family'].isin(['non_ml_rule_engine', 'not_ready_no_action'])
framed[['scenario_id', 'title', 'task_family', 'ml_suitable']]

In [ ]:
expected = {
    'S01': 'classification', 'S02': 'regression', 'S03': 'clustering',
    'S04': 'regression', 'S05': 'non_ml_rule_engine', 'S06': 'generation',
    'S07': 'not_ready_no_action', 'S08': 'ranking',
}
actual = framed.set_index('scenario_id')['task_family'].to_dict()
assert actual == expected
assert framed['scenario_id'].is_unique
print('PASS — 8/8 scenario task families.')

## 2. Decision-first framing

Chuỗi bắt buộc: **non-ML goal → decision → action → prediction → target → metric**. Prediction không thay đổi decision nào thì chỉ là phân tích, chưa phải ML product.

In [ ]:
four_frames = pd.DataFrame([
    {
        'case': 'support_triage',
        'non_ml_goal': 'reduce unprepared escalations without exceeding specialist capacity',
        'decision_action': 'route one new ticket to specialist or standard queue',
        'unit_time': 'one ticket; after parsing and before queue assignment',
        'target_output': 'escalation in 48h; binary probability',
        'metric_baseline': '10*FP + 200*FN; majority + current heuristic',
        'decision': 'conditional_go',
    },
    {
        'case': 'delivery_eta',
        'non_ml_goal': 'reduce promise misses at checkout',
        'decision_action': 'show ETA and choose a promise band',
        'unit_time': 'one order; checkout confirmation',
        'target_output': 'delivery duration; numeric hours',
        'metric_baseline': 'MAE/P90 error; route median',
        'decision': 'go_if_labels_reliable',
    },
    {
        'case': 'invoice_tax',
        'non_ml_goal': 'calculate auditable tax correctly',
        'decision_action': 'charge tax for each invoice line',
        'unit_time': 'one invoice line; invoice creation',
        'target_output': 'tax amount from deterministic rules',
        'metric_baseline': 'versioned rule-engine tests',
        'decision': 'no_go_ml_use_rules',
    },
    {
        'case': 'customer_discovery',
        'non_ml_goal': 'select cohorts for qualitative research',
        'decision_action': 'researcher chooses cohorts to interview',
        'unit_time': 'one customer-month; monthly batch',
        'target_output': 'no label; cluster assignment',
        'metric_baseline': 'stability + actionability; manual segmentation',
        'decision': 'exploratory_go',
    },
])
display(four_frames)
assert len(four_frames) == 4
assert four_frames['decision_action'].str.len().gt(0).all()

## 3. Canvas và prediction contract

Contract dưới đây khóa một prediction/ticket, thời điểm trước queue assignment, label trong 48 giờ và latency 500 ms. Chi phí chỉ là giả định đào tạo; production cần stakeholder xác nhận.

In [ ]:
contract = {
    'unit_of_prediction': 'one newly created support ticket',
    'prediction_time': 'after initial message parsing and before queue assignment',
    'target_name': 'escalated_within_48h',
    'target_definition': '1 if manager-confirmed escalation occurs within 48 hours; else 0',
    'target_window_start_hours': 0,
    'target_window_end_hours': 48,
    'output_type': 'binary',
    'latency_sla_ms': 500,
    'owner': 'Support Operations Manager',
}

canvas = {
    'problem_name': 'Early support escalation triage',
    'non_ml_goal': 'Reduce unprepared escalations without exceeding specialist capacity',
    'decision': 'Route each new ticket to specialist or standard queue',
    'decision_owner': 'Support Operations Manager',
    'prediction_unit': 'one newly created support ticket',
    'prediction_time': 'after initial parsing and before queue assignment',
    'target': 'escalated_within_48h',
    'target_window': '0 to 48 hours after ticket creation',
    'task_type': 'classification',
    'model_output': 'probability of escalation within 48 hours',
    'primary_metric': '10*FP + 200*FN on chronological holdout',
    'baseline': 'majority class and current triage_risk_score heuristic',
    'success_criterion': '15% lower cost than heuristic, recall >= 0.80, and predicted-positive rate <= 0.20',
    'deployment_mode': 'human_in_loop',
    'constraints': ['500 ms latency', 'specialist queue <= 20% per day'],
    'evaluation_slices': ['channel', 'customer_tier', 'outage_active'],
    'leakage_risks': ['post-outcome fields', 'future aggregates'],
    'non_goals': ['automatic ticket closure', 'agent performance scoring'],
}
display(pd.Series(contract, name='value').to_frame())

In [ ]:
canvas_required = {
    'problem_name', 'non_ml_goal', 'decision', 'decision_owner', 'prediction_unit',
    'prediction_time', 'target', 'target_window', 'task_type', 'model_output',
    'primary_metric', 'baseline', 'success_criterion', 'deployment_mode',
    'constraints', 'evaluation_slices', 'leakage_risks', 'non_goals',
}
contract_required = {
    'unit_of_prediction', 'prediction_time', 'target_name', 'target_definition',
    'target_window_start_hours', 'target_window_end_hours', 'output_type',
    'latency_sla_ms', 'owner',
}
assert canvas_required <= set(canvas)
assert all(canvas[key] for key in canvas_required)
assert contract_required <= set(contract)
assert 0 <= contract['target_window_start_hours'] < contract['target_window_end_hours']
assert contract['latency_sla_ms'] > 0
print('PASS — Canvas and prediction contract are structurally complete.')

## 4. Feature availability và leakage red-team

Chỉ feature `intended_for_model=True` mới được audit như input model mới. Baseline score được đánh giá riêng. Với support, parsing được phép hoàn tất trong 0.1 giờ.

In [ ]:
catalog = pd.read_csv(DATA / 'support_feature_catalog.csv')
allowed_offset_hours = 0.1
risks = []
for _, row in catalog.iterrows():
    intended = str(row['intended_for_model']).lower() == 'true'
    if not intended:
        continue
    stage = str(row['source_stage']).lower()
    if stage in {'post_outcome', 'label_generation'}:
        risks.append({
            'feature_name': row['feature_name'],
            'risk_type': 'post_outcome_leakage',
            'reason': f'source_stage={stage}',
        })
    elif float(row['available_offset_hours']) > allowed_offset_hours:
        risks.append({
            'feature_name': row['feature_name'],
            'risk_type': 'unavailable_at_prediction',
            'reason': f"available +{row['available_offset_hours']}h",
        })
risk_table = pd.DataFrame(risks)
display(risk_table)
assert set(risk_table['feature_name']) == {
    'resolution_hours', 'final_satisfaction', 'manager_override'
}
print('PASS — all three planted post-outcome fields detected.')

In [ ]:
framed.to_csv(OUTPUT / 'scenario_framing.csv', index=False)
four_frames.to_csv(OUTPUT / 'four_problem_frames.csv', index=False)
risk_table.to_csv(OUTPUT / 'support_feature_risks.csv', index=False)
with (OUTPUT / 'support_problem_canvas.json').open('w', encoding='utf-8') as file:
    json.dump(canvas, file, ensure_ascii=False, indent=2)
with (OUTPUT / 'support_prediction_contract.json').open('w', encoding='utf-8') as file:
    json.dump(contract, file, ensure_ascii=False, indent=2)

expected_files = [
    'scenario_framing.csv', 'four_problem_frames.csv', 'support_feature_risks.csv',
    'support_problem_canvas.json', 'support_prediction_contract.json',
]
assert all((OUTPUT / name).exists() for name in expected_files)
print('PASS — saved:', ', '.join(expected_files))

## Exit ticket

1. Feature nguy hiểm nhất và prediction-time evidence cần có là gì?  
2. Vì sao support canvas chỉ là Conditional Go?  
3. Viết một thay đổi có thể làm contract phải tăng version.